In [ ]:
# ============================================================================
# B__02-03-2026__ MODEL-ARCH.ipynb
#  All 5 Model Architectures + Loss Functions 
# Models: EfficientNet-B5 | InceptionV3 | ConvNeXtV2-Tiny | DenseNet201 | ResNeXt
# ============================================================================



---

## CELL 1: IMPORTS



In [ ]:
# ============================================================================
# CELL 1: IMPORTS
# ============================================================================

import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_properties(0).name}")



---

## CELL 2: SHARED CONFIGURATION



In [ ]:
# ============================================================================
# CELL 2: SHARED CONFIGURATION (Common across all 5 models)
# ============================================================================

# ── Class Labels ────────────────────────────────────────────────────────────
TARGET_LABELS = [
    'Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema',
    'Enlarged Cardiomediastinum', 'Fracture', 'Lung Lesion',
    'Lung Opacity', 'No Finding', 'Pleural Effusion', 'Pleural Other',
    'Pneumonia', 'Pneumothorax', 'Support Devices'
]


NUM_CLASSES     = len(TARGET_LABELS)            # 14
HARD_CLASSES    = ['Pneumothorax', 'Lung Lesion', 'Pleural Other', 'Fracture']

# ── Loss Hyper-parameters ───────────────────────────────────────────────────
USE_ADAPTIVE_GAMMA  = True
ASL_GAMMA_NEG       = 4.0
ASL_GAMMA_POS       = 1.0
BASE_GAMMA          = 2.0
HARD_CLASS_GAMMA    = 3.5
LABEL_SMOOTHING     = 0.1
INITIAL_TEMPERATURE = 1.5

# ── CBAM Parameters ─────────────────────────────────────────────────────────
USE_CBAM_ATTENTION       = True
CBAM_REDUCTION           = 16
CBAM_KERNEL_SIZE         = 7
USE_GRADIENT_CHECKPOINTING = True
USE_CHANNELS_LAST        = True

# ── Per-Model Image Sizes ────────────────────────────────────────────────────
MODEL_CONFIGS = {
    'efficientnet_b5'  : {'img_size': 456, 'dropout': 0.4},
    'inception_v3'     : {'img_size': 299, 'dropout': 0.4},
    'convnextv2_tiny'  : {'img_size': 224, 'dropout': 0.4},
    'densenet201'      : {'img_size': 256, 'dropout': 0.4},
    'resnext50_32x4d'  : {'img_size': 256, 'dropout': 0.4},   # ResNeXt
}

print("✅ Shared configuration loaded")
print(f"   Classes     : {NUM_CLASSES}")
print(f"   Hard classes: {HARD_CLASSES}")
print(f"\n   Per-model image sizes:")
for name, cfg in MODEL_CONFIGS.items():
    print(f"     {name:<22}: {cfg['img_size']}×{cfg['img_size']}")



---

## CELL 3: SHARED BUILDING BLOCKS (CBAM + EMA)



In [ ]:
# ============================================================================
# CELL 3: SHARED BUILDING BLOCKS
#   • ChannelAttention   – SE-style squeeze-and-excitation
#   • SpatialAttention   – conv-based spatial gate
#   • CBAM               – Channel + Spatial combined
#   • EMAModel           – Exponential Moving Average wrapper
#
# ============================================================================




# ── 3-A  Channel Attention ───────────────────────────────────────────────────
class ChannelAttention(nn.Module):
    """
    Squeeze-and-Excitation style channel attention.
    Learns WHAT feature channels to emphasise.
    Input  : (B, C, H, W)
    Output : (B, C, H, W)  ← element-wise scaled by sigmoid gate
    """
    def __init__(self, in_channels: int, reduction: int = 16):
        super().__init__()
        
        reduced = max(in_channels // reduction, 1)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(in_channels, reduced,    bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(reduced,    in_channels, bias=False),
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        b, c = x.size(0), x.size(1)
        avg = self.fc(self.avg_pool(x).view(b, c))
        mx  = self.fc(self.max_pool(x).view(b, c))
        return self.sigmoid((avg + mx).view(b, c, 1, 1))




# ── 3-B  Spatial Attention ───────────────────────────────────────────────────
class SpatialAttention(nn.Module):
    """
    Convolution-based spatial attention.
    Learns WHERE in the feature map to focus.
    Input  : (B, C, H, W)
    Output : (B, 1, H, W)  ← spatial gate
    """
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        padding = kernel_size // 2
        self.conv    = nn.Conv2d(2, 1, kernel_size, padding=padding, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = torch.mean(x, dim=1, keepdim=True)
        mx, _ = torch.max(x, dim=1, keepdim=True)
        return self.sigmoid(self.conv(torch.cat([avg, mx], dim=1)))




# ── 3-C  CBAM ────────────────────────────────────────────────────────────────
class CBAM(nn.Module):
    """
    Convolutional Block Attention Module (Woo et al., ECCV 2018).
    Sequentially applies Channel Attention → Spatial Attention.
    Input  : (B, C, H, W)
    Output : (B, C, H, W)  ← refined feature map
    """
    def __init__(self,
                 in_channels : int,
                 reduction   : int = 16,
                 kernel_size : int = 7):
        super().__init__()
        self.channel_att = ChannelAttention(in_channels, reduction)
        self.spatial_att = SpatialAttention(kernel_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x * self.channel_att(x)   # channel gate
        x = x * self.spatial_att(x)   # spatial gate
        return x




# ── 3-D  EMA Model ───────────────────────────────────────────────────────────
class EMAModel:
    """
    Exponential Moving Average of trainable model weights.
    Shared by all 5 training pipelines.
    Usage:
        ema = EMAModel(model, decay=0.9995)
        # after each optimiser step:
        ema.update(model)
        # for validation:
        ema.apply_shadow(model)
        ... validate ...
        ema.restore(model)
    """
    def __init__(self, model: nn.Module, decay: float = 0.9995):
        self.decay  = decay
        self.shadow = {n: p.data.clone()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.backup : dict = {}

    def update(self, model: nn.Module):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.data, alpha=1 - self.decay)

    def apply_shadow(self, model: nn.Module):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])

    def restore(self, model: nn.Module):
        for n, p in model.named_parameters():
            if p.requires_grad:
                p.data.copy_(self.backup[n])
        self.backup = {}


print("✅ Shared building blocks defined:")
print("   • ChannelAttention  – SE-style channel gate")
print("   • SpatialAttention  – conv spatial gate")
print("   • CBAM              – Channel + Spatial combined")
print("   • EMAModel          – Exponential Moving Average")



---

## CELL 4: LOSS FUNCTIONS



In [ ]:
# ============================================================================
# CELL 4: LOSS FUNCTIONS
#   Shared by ALL 5 models (identical across every notebook).
#
#   ① AsymmetricLoss   – different focal weights for pos/neg samples
#   ② CombinedLoss     – ASL  +  Adaptive Focal Loss  +  Label Smoothing
#
# Reference: "Asymmetric Loss For Multi-Label Classification" (ICCV 2021)
# ============================================================================


# ── 4-A  Asymmetric Loss ─────────────────────────────────────────────────────
class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss for Multi-Label Classification.

    Key Idea
    --------
    Use *different* focusing exponents for positive vs. negative samples:
      • gamma_pos (low ) → standard handling of true positives
      • gamma_neg (high) → aggressively down-weight easy negatives
                           ⟹ improves SPECIFICITY while keeping SENSITIVITY
    Optional probability shift (clip) further shifts the negative probability
    margin to reduce the contribution of very-easy negatives.
    Forward
    -------
    x : raw logits  (B, C)
    y : binary labels (B, C)   float in {0, 1}
    """
    def __init__(self,
                 gamma_neg : float = 4.0,
                 gamma_pos : float = 1.0,
                 clip      : float = 0.05,
                 eps       : float = 1e-8,
                 disable_torch_grad_focal_loss: bool = True):
        super().__init__()
        self.gamma_neg  = gamma_neg
        self.gamma_pos  = gamma_pos
        self.clip       = clip
        self.eps        = eps
        self.disable_torch_grad_focal_loss = disable_torch_grad_focal_loss


    def forward(self, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
        p      = torch.sigmoid(x)
        p_pos  = p
        p_neg  = 1 - p

        # ── Asymmetric probability shift (negatives only) ───────────────────
        if self.clip is not None and self.clip > 0:
            p_neg = (p_neg + self.clip).clamp(max=1.0)

        # ── Basic BCE ────────────────────────────────────────────────────────
        loss = (  y       * torch.log(p_pos.clamp(min=self.eps))
                + (1 - y) * torch.log(p_neg.clamp(min=self.eps)) )

        # ── Asymmetric Focal Weighting ───────────────────────────────────────
        if self.gamma_neg > 0 or self.gamma_pos > 0:
            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(False)

            pt         = p_pos * y + p_neg * (1 - y)
            gamma_map  = self.gamma_pos * y + self.gamma_neg * (1 - y)
            focal_w    = torch.pow(1 - pt, gamma_map)

            if self.disable_torch_grad_focal_loss:
                torch.set_grad_enabled(True)

            loss = loss * focal_w

        return -loss.mean()




# ── 4-B  Combined Loss ───────────────────────────────────────────────────────
class CombinedLoss(nn.Module):
    """
    Production loss used by ALL 5 models.

    Components
    ----------
    1. AsymmetricLoss   (weight = 0.5)  → better class-level specificity
    2. Adaptive Focal Loss (weight = 0.5)
         • per-class gamma: BASE_GAMMA for normal classes
                            HARD_CLASS_GAMMA for rare/hard classes
         • uses pos_weight for class-imbalance correction
    3. Label smoothing applied to the focal BCE term only

    Parameters
    ----------
    alpha          : focal scaling factor          (default 0.25)
    base_gamma     : default focal exponent        (default 2.0)
    hard_gamma     : focal exponent for hard classes (default 3.5)
    asl_gamma_neg  : gamma for negatives in ASL    (default 4.0)
    asl_gamma_pos  : gamma for positives in ASL    (default 1.0)
    pos_weight     : per-class BCE pos_weight tensor (None → uniform)
    label_smoothing: label smoothing factor        (default 0.1)
    """

    def __init__(self,
                 alpha          : float              = 0.25,
                 base_gamma     : float              = 2.0,
                 hard_gamma     : float              = 3.5,
                 asl_gamma_neg  : float              = 4.0,
                 asl_gamma_pos  : float              = 1.0,
                 pos_weight     : torch.Tensor | None = None,
                 label_smoothing: float              = 0.1):
        super().__init__()

        # Fixed 50/50 blend  (no learnable params → stable training)
        self.asl_weight   = 0.5
        self.focal_weight = 0.5

        # ── ASL sub-loss ─────────────────────────────────────────────────────
        self.asl = AsymmetricLoss(
            gamma_neg=asl_gamma_neg,
            gamma_pos=asl_gamma_pos,
            clip=0.05,
        )

        # ── Focal sub-loss params ────────────────────────────────────────────
        self.alpha          = alpha
        self.pos_weight     = pos_weight
        self.label_smoothing = label_smoothing

        # ── Per-class gamma buffer ────────────────────────────────────────────
        gammas = torch.ones(NUM_CLASSES) * base_gamma
        if USE_ADAPTIVE_GAMMA:
            for i, label in enumerate(TARGET_LABELS):
                if label in HARD_CLASSES:
                    gammas[i] = hard_gamma
        self.register_buffer('class_gammas', gammas)




    def forward(self,
                logits : torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        """
        logits  : (B, 14) raw model output
        targets : (B, 14) ground-truth binary labels
        """
        # ── Label smoothing ───────────────────────────────────────────────────
        t_smooth = targets * (1 - self.label_smoothing) + self.label_smoothing / 2

        # ── Component 1 : ASL ─────────────────────────────────────────────────
        asl_loss = self.asl(logits, targets)

        # ── Component 2 : Adaptive Focal ─────────────────────────────────────
        bce = F.binary_cross_entropy_with_logits(
            logits, t_smooth,
            pos_weight=self.pos_weight,
            reduction='none',
        )                                             # (B, 14)
        pt          = torch.exp(-bce)                 # predicted probability
        gamma       = self.class_gammas.to(logits.device).unsqueeze(0)
        focal_w     = self.alpha * (1 - pt).pow(gamma)
        focal_loss  = (focal_w * bce).mean()

        return self.asl_weight * asl_loss + self.focal_weight * focal_loss




# ── Quick sanity check ────────────────────────────────────────────────────────
_logits  = torch.randn(4, NUM_CLASSES)
_targets = torch.randint(0, 2, (4, NUM_CLASSES)).float()
_loss_fn = CombinedLoss()
_out     = _loss_fn(_logits, _targets)
del _logits, _targets, _loss_fn, _out

print("✅ Loss Functions Defined & Verified:")
print("   ① AsymmetricLoss")
print(f"      gamma_neg={ASL_GAMMA_NEG}  gamma_pos={ASL_GAMMA_POS}  clip=0.05")
print("   ② CombinedLoss  (ASL × 0.5  +  AdaptiveFocal × 0.5)")
print(f"      base_gamma={BASE_GAMMA}  hard_gamma={HARD_CLASS_GAMMA}")
print(f"      label_smoothing={LABEL_SMOOTHING}")
print(f"      hard classes → {HARD_CLASSES}")



---

## CELL 5: MODEL 1 — EfficientNet-B5



In [ ]:
# ============================================================================
# CELL 5: MODEL 1 – EfficientNet-B5
# ============================================================================
#
#  Backbone  : EfficientNet-B5  (timm, pretrained ImageNet)
#  Image size: 456 × 456   (native B5 resolution)
#  Params    : ~30 M
#
#  Feature stages (features_only=True):
#  ┌────────┬──────────┬──────────────┬─────────────────────────┐
#  │ Stage  │ Channels │ Spatial size │ Role                    │
#  ├────────┼──────────┼──────────────┼─────────────────────────┤
#  │  0     │   24     │  H/8         │ low-level  (skip)       │
#  │  1     │   40     │  H/16        │ textures   (skip)       │
#  │  2     │   64     │  H/32        │ mid-level  (skip)       │
#  │  3     │  176     │  H/64        │ regions    → CBAM ✓     │
#  │  4     │  512     │  H/128       │ semantics  → CBAM ✓     │
#  └────────┴──────────┴──────────────┴─────────────────────────┘
#
#  Fusion head: stages [2, 3, 4]  →  concat(512×3)  →  Linear→512
# ============================================================================


class EfficientMultiScaleFusion(nn.Module):
    """
    Fuses 3 EfficientNet-B5 feature stages into a single 512-d vector.
    Each stage is: AdaptiveAvgPool → Flatten → Linear(→512) → SiLU → Dropout
    Then concat(3 × 512) → Linear(1536→512) → SiLU → Dropout
    """
    def __init__(self, in_dims: list, out_dim: int = 512):
        super().__init__()

        def _proj(d):
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(d, out_dim), 
                nn.SiLU(), 
                nn.Dropout(0.2),
            )

        self.proj2   = _proj(in_dims[2])
        self.proj3   = _proj(in_dims[3])
        self.proj4   = _proj(in_dims[4])

        self.fusion  = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim), 
            nn.SiLU(), 
            nn.Dropout(0.3),
        )


    def forward(self, feats: list) -> torch.Tensor:
        p2 = self.proj2(feats[2])
        p3 = self.proj3(feats[3])
        p4 = self.proj4(feats[4])
        out = self.fusion(torch.cat([p2, p3, p4], dim=1))
        return out




class EfficientNetB5Model(nn.Module):
    """EfficientNet-B5 with CBAM on stages 3 & 4 + multi-scale fusion."""
    def __init__(self,
                 model_name  = 'efficientnet_b5',
                 num_classes : int   = NUM_CLASSES,
                 dropout     : float = 0.4,
                 img_size    : int   = 456):
        super().__init__()
        self.img_size = img_size

        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            model_name, pretrained=True,
            features_only=True, num_classes=0,
            drop_rate=dropout, drop_path_rate=0.2,
        )

        if USE_GRADIENT_CHECKPOINTING and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)

        # ── Discover stage dims ───────────────────────────────────────────────
        with torch.no_grad():
            dummy  = torch.randn(1, 3, img_size, img_size)
            feats  = self.backbone(dummy)
            self.stage_dims = [f.shape[1] for f in feats]   # [24,40,64,176,512]
            del dummy, feats

        # ── CBAM on deep stages ───────────────────────────────────────────────
        self.cbam3 = CBAM(self.stage_dims[3], CBAM_REDUCTION, CBAM_KERNEL_SIZE)
        self.cbam4 = CBAM(self.stage_dims[4], CBAM_REDUCTION, CBAM_KERNEL_SIZE)

        # ── Fusion + classifier ───────────────────────────────────────────────
        self.fusion_head = EfficientMultiScaleFusion(self.stage_dims, out_dim=512)

        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), 
            nn.Linear(512, num_classes),
        )
        
        self.temperature = nn.Parameter(torch.ones(1) * INITIAL_TEMPERATURE)



    def forward(self,x : torch.Tensor,return_attention : bool = False):

        feats = self.backbone(x)
        feats[3] = self.cbam3(feats[3])
        feats[4] = self.cbam4(feats[4])
        fused    = self.fusion_head(feats)
        logits   = self.classifier(fused)

        if return_attention:
            return logits, {
                'stage3': feats[3].mean(1, keepdim=True),
                'stage4': feats[4].mean(1, keepdim=True),
            }
        return logits


# ── Summary ───────────────────────────────────────────────────────────────────
def _print_model_summary(model, name, img_size):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n{'─'*55}")
    print(f"  {name}")
    print(f"{'─'*55}")
    print(f"  Total params    : {total/1e6:.2f} M")
    print(f"  Trainable params: {trainable/1e6:.2f} M")
    print(f"  Input size      : {img_size}×{img_size}")

_m = EfficientNetB5Model()
_print_model_summary(_m, "MODEL 1 – EfficientNet-B5", 456)
del _m
torch.cuda.empty_cache()
print("✅ EfficientNetB5Model defined")



---

## CELL 6: MODEL 2 — Inception-V3



In [ ]:
# ============================================================================
# CELL 6: MODEL 2 – Inception-V3
# ============================================================================
#
#  Backbone  : InceptionV3  (timm, pretrained ImageNet)
#  Image size: 299 × 299   (native IV3 resolution)
#  Params    : ~23-27 M
#
#  Feature stages (features_only=True):
#  ┌────────┬──────────┬──────────────┬─────────────────────────┐
#  │ Stage  │ Channels │ Spatial size │ Role                    │
#  ├────────┼──────────┼──────────────┼─────────────────────────┤
#  │  0     │   64     │  H/8   37×37 │ stem      (skip)        │
#  │  1     │  256     │  H/8   37×37 │ Inception A (skip)      │
#  │  2     │  768     │  H/16  19×19 │ Inception B → Fusion ✓  │
#  │  3     │ 2048     │  H/32  10×10 │ Inception C → CBAM ✓    │
#  └────────┴──────────┴──────────────┴─────────────────────────┘
#
#  Fusion head: stages [2, 3]  →  concat(512×2)  →  Linear→512
# ============================================================================




class InceptionV3MultiScaleFusion(nn.Module):
    """
    Fuses Inception-V3 stages 2 & 3 into a 512-d vector.
    Each branch: AdaptiveAvgPool → Flatten → Linear(→512) → BN → GELU
    Concat(1024) → Linear(→512) → BN → GELU → Dropout
    """
    def __init__(self, in_dims: list, out_dim: int = 512):
        super().__init__()

        def _proj(d):
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(d, out_dim, bias=False),
                nn.BatchNorm1d(out_dim), nn.GELU(),
            )

        self.proj2  = _proj(in_dims[2])
        self.proj3  = _proj(in_dims[3])
        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 2, out_dim, bias=False),
            nn.BatchNorm1d(out_dim), nn.GELU(), nn.Dropout(0.3),
        )

    def forward(self, feats: list) -> torch.Tensor:
        return self.fusion(torch.cat([self.proj2(feats[2]),
                                      self.proj3(feats[3])], dim=1))


class InceptionV3Model(nn.Module):
    """Inception-V3 with CBAM on stage 3 + 2-stage fusion."""

    def __init__(self,
                 model_name  = 'inception_v3',
                 num_classes : int   = NUM_CLASSES,
                 dropout     : float = 0.4,
                 img_size    : int   = 299):
        super().__init__()
        self.img_size = img_size

        # ── Backbone (CPU init to avoid device issues) ────────────────────────
        self.backbone = timm.create_model(
            model_name, pretrained=True,
            features_only=True, num_classes=0, drop_rate=dropout,
        )

        # ── Stage dims ────────────────────────────────────────────────────────
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            feats = self.backbone(dummy)
            self.stage_dims = [f.shape[1] for f in feats]   # [64,256,768,2048]
            del dummy, feats

        # ── CBAM on deepest stage ─────────────────────────────────────────────
        self.cbam3 = CBAM(self.stage_dims[3], CBAM_REDUCTION, CBAM_KERNEL_SIZE)

        # ── Fusion + classifier ───────────────────────────────────────────────
        self.fusion_head = InceptionV3MultiScaleFusion(self.stage_dims, out_dim=512)
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(512, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1) * INITIAL_TEMPERATURE)
        self._checkpoint_enabled = False

    def enable_gradient_checkpointing(self):
        if USE_GRADIENT_CHECKPOINTING and not self._checkpoint_enabled:
            if hasattr(self.backbone, 'set_grad_checkpointing'):
                try:
                    self.backbone.set_grad_checkpointing(enable=True)
                    self._checkpoint_enabled = True
                except Exception:
                    pass

    def forward(self,
                x                : torch.Tensor,
                return_attention : bool = False):
        feats    = self.backbone(x)
        feats[3] = self.cbam3(feats[3])
        fused    = self.fusion_head(feats)
        logits   = self.classifier(fused)

        if return_attention:
            return logits, {'stage3': feats[3].mean(1, keepdim=True)}
        return logits


_m = InceptionV3Model()
_print_model_summary(_m, "MODEL 2 – Inception-V3", 299)
del _m
torch.cuda.empty_cache()
print("✅ InceptionV3Model defined")



---

## CELL 7: MODEL 3 — ConvNeXtV2-Tiny



In [ ]:
# ============================================================================
# CELL 7: MODEL 3 – ConvNeXtV2-Tiny
# ============================================================================
#
#  Backbone  : ConvNeXtV2 Tiny  (timm, pretrained ImageNet)
#  Image size: 224 × 224  (native resolution; 6× faster than 456)
#  Params    : ~28 M
#
#  Feature stages (features_only=True):
#  ┌────────┬──────────┬──────────────┬─────────────────────────┐
#  │ Stage  │ Channels │ Spatial size │ Role                    │
#  ├────────┼──────────┼──────────────┼─────────────────────────┤
#  │  0     │   96     │  H/4  56×56  │ stem      (skip)        │
#  │  1     │  192     │  H/8  28×28  │ Stage 1   → Fusion ✓    │
#  │  2     │  384     │  H/16 14×14  │ Stage 2   → CBAM ✓      │
#  │  3     │  768     │  H/32  7×7   │ Stage 3   → CBAM ✓      │
#  └────────┴──────────┴──────────────┴─────────────────────────┘
#
#  Fusion head: stages [1, 2, 3]  →  concat(512×3)  →  Linear→512
# ============================================================================


class ConvNeXtV2MultiScaleFusion(nn.Module):
    """
    Fuses ConvNeXtV2-Tiny stages 1, 2, 3 into a 512-d vector.
    Each branch: AdaptiveAvgPool → Flatten → Linear(→512) → BN → GELU
    Concat(1536) → Linear(→512) → BN → GELU → Dropout
    """
    def __init__(self, in_dims: list, out_dim: int = 512):
        super().__init__()

        def _proj(d):
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(d, out_dim, bias=False),
                nn.BatchNorm1d(out_dim), nn.GELU(),
            )

        self.proj1  = _proj(in_dims[0])
        self.proj2  = _proj(in_dims[1])
        self.proj3  = _proj(in_dims[2])
        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim, bias=False),
            nn.BatchNorm1d(out_dim), 
            nn.GELU(), 
            nn.Dropout(0.3),
        )

    def forward(self, feats: list) -> torch.Tensor:
        """feats = [stage1, stage2, stage3] (stage0 already excluded upstream)"""
        return self.fusion(torch.cat([self.proj1(feats[0]),
                                      self.proj2(feats[1]),
                                      self.proj3(feats[2])], dim=1))





class ConvNeXtV2Model(nn.Module):
    """ConvNeXtV2-Tiny with CBAM on stages 2 & 3 + 3-stage fusion."""

    def __init__(self,
                 model_name  = 'convnextv2_tiny',
                 num_classes : int   = NUM_CLASSES,
                 dropout     : float = 0.4,
                 img_size    : int   = 224):
        super().__init__()
        self.img_size = img_size

        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True,
            features_only=True, 
            num_classes=0,
            drop_rate=dropout, 
            drop_path_rate=0.2,
        )

        if USE_GRADIENT_CHECKPOINTING and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)

        # ── Stage dims  (skip stage 0 → use stages 1,2,3) ───────────────────
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            feats = self.backbone(dummy)
            all_dims  = [f.shape[1] for f in feats]   # [96,192,384,768]
            self.stage_dims = all_dims[1:4]                 # [192, 384, 768]
            del dummy, feats

        # ── CBAM on stages 2 & 3 ─────────────────────────────────────────────
        self.cbam2 = CBAM(self.stage_dims[1], CBAM_REDUCTION, CBAM_KERNEL_SIZE)
        self.cbam3 = CBAM(self.stage_dims[2], CBAM_REDUCTION, CBAM_KERNEL_SIZE)

        # ── Fusion + classifier ───────────────────────────────────────────────
        self.fusion_head = ConvNeXtV2MultiScaleFusion(self.stage_dims, out_dim=512)
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(512, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1) * INITIAL_TEMPERATURE)



    def forward(self,
                x                : torch.Tensor,
                return_attention : bool = False):
        all_feats     = self.backbone(x)           # 4 tensors
        s1, s2, s3    = all_feats[1], all_feats[2], all_feats[3]

        s2         = self.cbam2(s2)
        s3         = self.cbam3(s3)
        fused      = self.fusion_head([s1, s2, s3])
        logits     = self.classifier(fused)

        if return_attention:
            return logits, {
                'stage2': s2.mean(1, keepdim=True),
                'stage3': s3.mean(1, keepdim=True),
            }
        return logits




_m = ConvNeXtV2Model()
_print_model_summary(_m, "MODEL 3 – ConvNeXtV2-Tiny", 224)
del _m
torch.cuda.empty_cache()
print("✅ ConvNeXtV2Model defined")



---

## CELL 8: MODEL 4 — DenseNet-201



In [ ]:
# ============================================================================
# CELL 8: MODEL 4 – DenseNet-201
# ============================================================================
#
#  Backbone  : DenseNet201  (timm, pretrained ImageNet)
#  Image size: 256 × 256
#  Params    : ~20 M  (most efficient in this suite)
#
#  Feature stages (features_only=True  →  4 dense blocks):
#  ┌──────────────┬──────────┬──────────────┬─────────────────────────┐
#  │ Dense Block  │ Channels │ Spatial size │ Role                    │
#  ├──────────────┼──────────┼──────────────┼─────────────────────────┤
#  │  Block 1     │  256     │  H/4         │ low-level  (skip)       │
#  │  Block 2     │  512     │  H/8         │ mid-level  → Fusion ✓   │
#  │  Block 3     │ 1792     │  H/16        │ deep       → CBAM ✓     │
#  │  Block 4     │ 1920     │  H/32        │ deepest    → CBAM ✓     │
#  └──────────────┴──────────┴──────────────┴─────────────────────────┘
#
#  Fusion head: blocks [2, 3, 4]  →  concat(512×3)  →  Linear→512
# ============================================================================



class DenseNetMultiScaleFusion(nn.Module):
    """
    Fuses DenseNet-201 blocks 2, 3, 4 into a 512-d vector.
    Each branch: AdaptiveAvgPool → Flatten → Linear(→512) → SiLU → Dropout
    Concat(1536) → Linear(→512) → SiLU → Dropout
    """
    def __init__(self, in_dims: list, out_dim: int = 512):
        super().__init__()

        def _proj(d):
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1), 
                nn.Flatten(),
                nn.Linear(d, out_dim), 
                nn.SiLU(), 
                nn.Dropout(0.2),
            )

        self.proj1  = _proj(in_dims[0])   # block 2
        self.proj2  = _proj(in_dims[1])   # block 3
        self.proj3  = _proj(in_dims[2])   # block 4

        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim), 
            nn.SiLU(), 
            nn.Dropout(0.3),
        )

    def forward(self, feats: list) -> torch.Tensor:
        return self.fusion(torch.cat([self.proj1(feats[0]),
                                      self.proj2(feats[1]),
                                      self.proj3(feats[2])], dim=1))





class DenseNet201Model(nn.Module):
    """DenseNet-201 with CBAM on blocks 3 & 4 + 3-block fusion."""

    def __init__(self,
                 model_name  = 'densenet201',
                 num_classes : int   = NUM_CLASSES,
                 dropout     : float = 0.4,
                 img_size    : int   = 256):
        super().__init__()
        self.img_size = img_size

        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True,
            features_only=True, 
            num_classes=0, 
            drop_rate=dropout,
        )

        if USE_GRADIENT_CHECKPOINTING and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)

        # ── Dense block dims  (skip block 1 → use blocks 2,3,4) ─────────────
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            feats = self.backbone(dummy)
            all_dims = [f.shape[1] for f in feats]  # [256,512,1792,1920]
            self.block_dims   = all_dims[1:4]                 # [512, 1792, 1920]
            del dummy, feats


        # ── CBAM on deep blocks ────────────────────────────────────────────────
        self.cbam3 = CBAM(self.block_dims[1], CBAM_REDUCTION, CBAM_KERNEL_SIZE)
        self.cbam4 = CBAM(self.block_dims[2], CBAM_REDUCTION, CBAM_KERNEL_SIZE)

        # ── Fusion + classifier ───────────────────────────────────────────────
        self.fusion_head = DenseNetMultiScaleFusion(self.block_dims, out_dim=512)
        self.classifier  = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(512, num_classes),
        )
        self.temperature = nn.Parameter(torch.ones(1) * INITIAL_TEMPERATURE)




    def forward(self,
                x : torch.Tensor, 
                return_attention : bool = False):
        
        feats    = self.backbone(x)               # 4 tensors
        feats[2] = self.cbam3(feats[2])           # block 3
        feats[3] = self.cbam4(feats[3])           # block 4
        fused    = self.fusion_head(feats[1:])    # blocks 2,3,4
        logits   = self.classifier(fused)

        if return_attention:
            return logits, {
                'block3': feats[2].mean(1, keepdim=True),
                'block4': feats[3].mean(1, keepdim=True),
            }
        return logits


_m = DenseNet201Model()
_print_model_summary(_m, "MODEL 4 – DenseNet-201", 256)
del _m
torch.cuda.empty_cache()
print("✅ DenseNet201Model defined")



---

## CELL 9: MODEL 5 — ResNeXt-50 (32×4d)



In [ ]:
# ============================================================================
# CELL 9: MODEL 5 – ResNeXt-50 (32×4d)
# ============================================================================
#
#  Backbone  : ResNeXt-50 (32×4d)  (timm, pretrained ImageNet)
#  Image size: 256 × 256
#  Params    : ~25 M
#  Difference from ResNet-50: grouped convolutions (32 groups × 4d width)
#                             → better representational capacity, same FLOPs
#
#  Feature stages (features_only=True):
#  ┌────────┬──────────┬──────────────┬─────────────────────────┐
#  │ Stage  │ Channels │ Spatial size │ Role                    │
#  ├────────┼──────────┼──────────────┼─────────────────────────┤
#  │  0     │   64     │  H/4         │ stem      (skip)        │
#  │  1     │  256     │  H/4         │ layer1    (skip)        │
#  │  2     │  512     │  H/8         │ layer2    → Fusion ✓    │
#  │  3     │ 1024     │  H/16        │ layer3    → CBAM ✓      │
#  │  4     │ 2048     │  H/32        │ layer4    → CBAM ✓      │
#  └────────┴──────────┴──────────────┴─────────────────────────┘
#
#  Fusion head: stages [2, 3, 4]  →  concat(512×3)  →  Linear→512
# ============================================================================





class ResNeXtMultiScaleFusion(nn.Module):
    """
    Fuses ResNeXt-50 stages 2, 3, 4 into a 512-d vector.
    Each branch: AdaptiveAvgPool → Flatten → Linear(→512) → BN → ReLU → Dropout
    Concat(1536) → Linear(→512) → BN → ReLU → Dropout
    """
    def __init__(self, in_dims: list, out_dim: int = 512):
        super().__init__()

        def _proj(d):
            return nn.Sequential(
                nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                nn.Linear(d, out_dim, bias=False),
                nn.BatchNorm1d(out_dim), nn.ReLU(inplace=True), nn.Dropout(0.2),
            )

        self.proj2  = _proj(in_dims[0])   # stage 2 → 512 ch
        self.proj3  = _proj(in_dims[1])   # stage 3 → 1024 ch
        self.proj4  = _proj(in_dims[2])   # stage 4 → 2048 ch

        self.fusion = nn.Sequential(
            nn.Linear(out_dim * 3, out_dim, bias=False),
            nn.BatchNorm1d(out_dim), 
            nn.ReLU(inplace=True), 
            nn.Dropout(0.3),
        )

    def forward(self, feats: list) -> torch.Tensor:
        return self.fusion(torch.cat([self.proj2(feats[0]),
                                      self.proj3(feats[1]),
                                      self.proj4(feats[2])], dim=1))





class ResNeXtModel(nn.Module):
    """ResNeXt-50 (32×4d) with CBAM on stages 3 & 4 + 3-stage fusion."""

    def __init__(self,
                 model_name  = 'resnext50_32x4d',
                 num_classes : int   = NUM_CLASSES,
                 dropout     : float = 0.4,
                 img_size    : int   = 256):
        super().__init__()
        self.img_size = img_size

        # ── Backbone ──────────────────────────────────────────────────────────
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True,
            features_only=True, 
            num_classes=0, 
            drop_rate=dropout,
        )

        if USE_GRADIENT_CHECKPOINTING and hasattr(self.backbone, 'set_grad_checkpointing'):
            self.backbone.set_grad_checkpointing(enable=True)

        # ── Stage dims  (use stages 2,3,4) ───────────────────────────────────
        with torch.no_grad():
            dummy = torch.randn(1, 3, img_size, img_size)
            feats = self.backbone(dummy)
            all_dims = [f.shape[1] for f in feats]  # [64,256,512,1024,2048]
            self.stage_dims = all_dims[2:5]                 # [512, 1024, 2048]
            del dummy, feats


        # ── CBAM on deep stages ────────────────────────────────────────────────
        self.cbam3 = CBAM(self.stage_dims[1], CBAM_REDUCTION, CBAM_KERNEL_SIZE)
        self.cbam4 = CBAM(self.stage_dims[2], CBAM_REDUCTION, CBAM_KERNEL_SIZE)


        # ── Fusion + classifier ───────────────────────────────────────────────
        self.fusion_head = ResNeXtMultiScaleFusion(self.stage_dims, out_dim=512)
        self.classifier  = nn.Sequential(nn.Dropout(dropout), nn.Linear(512, num_classes))
        self.temperature = nn.Parameter(torch.ones(1) * INITIAL_TEMPERATURE)




    def forward(self,x : torch.Tensor,
                return_attention : bool = False):
        
        feats    = self.backbone(x)               # 5 tensors
        s2, s3, s4 = feats[2], feats[3], feats[4]

        s3     = self.cbam3(s3)
        s4     = self.cbam4(s4)
        fused  = self.fusion_head([s2, s3, s4])
        logits = self.classifier(fused)

        if return_attention:
            return logits, {
                'stage3': s3.mean(1, keepdim=True),
                'stage4': s4.mean(1, keepdim=True),
            }
        return logits


_m = ResNeXtModel()
_print_model_summary(_m, "MODEL 5 – ResNeXt-50 (32×4d)", 256)
del _m
torch.cuda.empty_cache()
print("✅ ResNeXtModel defined")



---

## CELL 10: MODEL FACTORY + COMPARISON TABLE



In [ ]:
# ============================================================================
# CELL 10: MODEL FACTORY + COMPARISON TABLE
# ============================================================================

MODEL_REGISTRY = {
    'efficientnet_b5' : EfficientNetB5Model,
    'inception_v3'    : InceptionV3Model,
    'convnextv2_tiny' : ConvNeXtV2Model,
    'densenet201'     : DenseNet201Model,
    'resnext50_32x4d' : ResNeXtModel,
}


def get_model(model_name: str) -> nn.Module:
    """
    Instantiate any of the 5 models by name, move to DEVICE,
    optionally apply channels-last memory format for GPU throughput.
    """
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown model: {model_name}. "
                         f"Choose from {list(MODEL_REGISTRY.keys())}")

    cfg   = MODEL_CONFIGS[model_name]
    klass = MODEL_REGISTRY[model_name]

    torch.cuda.empty_cache()
    gc.collect()

    model = klass(
        model_name  = model_name,
        num_classes = NUM_CLASSES,
        dropout     = cfg['dropout'],
        img_size    = cfg['img_size'],
    ).to(DEVICE)

    if USE_CHANNELS_LAST and DEVICE == 'cuda':
        model = model.to(memory_format=torch.channels_last)

    return model


# ── Comparison Table ──────────────────────────────────────────────────────────
print("\n" + "="*90)
print("  ARCHITECTURE COMPARISON TABLE")
print("="*90)

header = (f"{'Model':<22} {'Img':>5} {'Stages fused':<20} "
          f"{'CBAM stages':<18} {'~Params':>9}  Activation")
print(header)
print("-"*90)

info = [
    ('efficientnet_b5', '456²', 'stages [2,3,4]',    'stages 3,4', '~30.4 M', 'SiLU'),
    ('inception_v3',    '299²', 'stages [2,3]',      'stage 3',    '~24.0 M', 'ReLU'),
    ('convnextv2_tiny', '224²', 'stages [1,2,3]',    'stages 2,3', '~28.6 M', 'GELU'),
    ('densenet201',     '256²', 'blocks [2,3,4]',    'blocks 3,4', '~20.0 M', 'SiLU'),
    ('resnext50_32x4d', '256²', 'stages [2,3,4]',    'stages 3,4', '~25.0 M', 'ReLU'),
]

for name, img, fused, cbam, params, act in info:
    print(f"  {name:<20} {img:>5} {fused:<20} {cbam:<18} {params:>9}  {act}")

print("-"*90)
print("\n  Shared components across ALL 5 models:")
print("    • CBAM attention  (ChannelAttention + SpatialAttention)")
print("    • Multi-scale feature fusion head  →  512-d output")
print("    • Learnable temperature scaling    (INITIAL_TEMP = 1.5)")
print("    • CombinedLoss  (ASL × 0.5  +  AdaptiveFocal × 0.5)")
print("    • EMA weights  (decay = 0.9995)")
print("    • SWA  (Stochastic Weight Averaging, starts epoch 20)")
print("    • MixUp + CutMix data augmentation")
print("    • Cosine Annealing with Warm Restarts  (T0=8, Tmult=2)")
print("    • Multi-metric early stopping  (patience = 12)")